# ARC-v0.25 — FEVER-E5 Severity-Matched Validation

**Purpose.** Test whether the representation-versus-search-effort H3abs contrast survives after one-shot effectiveness severity is approximately matched on FIT.

Frozen FIT calibration:

- representation: **IVF-PQ32 @ nprobe=64 → IVF-SQ8 @ nprobe=64**
- search effort: **IVF-SQ8 @ nprobe=8 → IVF-SQ8 @ nprobe=64**
- FIT representation gap: **0.249338**
- FIT matched search-effort gap: **0.238035**
- relative mismatch: **4.533%**
- selected `nprobe_low = 8`
- frozen protocol SHA256: `62351de5e3d2993129fd0b4f67e3fd49e06b97059b050cd800645d2d6c467e24`

This notebook **does not** re-encode FEVER, rebuild FAISS indexes, or rerun the existing representation branch. It reuses v0.18 representation endpoints and runs only the frozen nprobe branch on the 3,316 validation queries.

The validation result is retained regardless of sign or significance. Do not change the matched nprobe, policy grid, endpoint definitions, or split after seeing validation trajectories.

In [ ]:
# Cell 1 — Install/imports and fixed paths
!pip -q install faiss-cpu pyarrow scipy tqdm

from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone
import hashlib, json, os, time

import faiss
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

SEED = 20260824
np.random.seed(SEED)

DIM = 384
TOP_RETRIEVE = 100
UTILITY_K = 10
MAX_ROUNDS = 4
HIGH_NPROBE = 64
MATCHED_NPROBE = 8
CHECKPOINT_EVERY_QUERIES = 10
BOOTSTRAP_REPS = 10_000

RUN_V018 = Path(
    "/content/drive/MyDrive/rag-pq-checkpoints/arc-v0/"
    "cross-encoder-fever-replication-v018/20260819-015645"
)

SPLIT_PATH = Path(
    "/content/drive/MyDrive/rag-pq-checkpoints/arc-v0/"
    "fever-boundary-external-replication-v013/20260817-151852/"
    "v013_boundary_query_split.csv"
)

OUT = Path(
    "/content/drive/MyDrive/rag-pq-checkpoints/arc-v0/"
    "fever-e5-severity-matched-mechanism-v025/20260824-095157"
)

RAW_FEVER = Path(
    "/content/drive/MyDrive/rag-pq-checkpoints/raw-datasets/fever"
)

EXPECTED_PROTOCOL_SHA = (
    "62351de5e3d2993129fd0b4f67e3fd49e06b97059b050cd800645d2d6c467e24"
)

faiss.omp_set_num_threads(os.cpu_count() or 1)

print("threads:", faiss.omp_get_max_threads())
print("v0.18:", RUN_V018)
print("v0.25:", OUT)

In [ ]:
# Cell 2 — Verify frozen v0.25 protocol and split identity
def sha256_file(path, chunk=16 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def membership_sha(ids):
    return hashlib.sha256(
        "\n".join(sorted(map(str, ids))).encode("utf-8")
    ).hexdigest()

PROTOCOL_PATH = OUT / "v025_frozen_severity_match_protocol.json"
assert PROTOCOL_PATH.is_file(), PROTOCOL_PATH

PROTOCOL_SHA = sha256_file(PROTOCOL_PATH)
assert PROTOCOL_SHA == EXPECTED_PROTOCOL_SHA, (PROTOCOL_SHA, EXPECTED_PROTOCOL_SHA)

protocol = json.loads(PROTOCOL_PATH.read_text())
assert int(protocol["selected_search_effort_low_nprobe"]) == MATCHED_NPROBE
assert protocol["calibration_split"] == "FIT only"
assert protocol["validation_trajectory_accessed"] is False
assert protocol["post_selection_retuning_allowed"] is False

split_df = pd.read_csv(SPLIT_PATH)
FIT_IDS = split_df.loc[split_df["split"].eq("fit"), "query_id"].astype(str).tolist()
VAL_IDS = split_df.loc[split_df["split"].eq("validation"), "query_id"].astype(str).tolist()

EXPECTED_FIT_SHA = "85c01de943636a080abe10cb18cf5538b705b89f6f042a74c92bac366d89612c"
EXPECTED_VAL_SHA = "5e7bd8e3e5e3f0120b1e93726a19183285624c405ef68b34005c8766a9568b42"

assert len(FIT_IDS) == 3350
assert len(VAL_IDS) == 3316
assert membership_sha(FIT_IDS) == EXPECTED_FIT_SHA
assert membership_sha(VAL_IDS) == EXPECTED_VAL_SHA
assert set(FIT_IDS).isdisjoint(VAL_IDS)

print("FROZEN PROTOCOL — PASS")
print("Protocol SHA:", PROTOCOL_SHA)
print("FIT:", len(FIT_IDS), "VAL:", len(VAL_IDS))
print("MATCHED_NPROBE:", MATCHED_NPROBE)

In [ ]:
# Cell 3 — Restore v0.18 query embeddings and FAISS indexes
QUERY_IDS_PATH = RUN_V018 / "dev_query_ids.txt"
QUERY_EMB_PATH = RUN_V018 / "dev_query_embeddings.float32.npy"

DEV_QUERY_IDS = QUERY_IDS_PATH.read_text().splitlines()
dev_query_embeddings = np.load(QUERY_EMB_PATH, mmap_mode="r")
DEV_QUERY_INDEX = {str(qid): i for i, qid in enumerate(DEV_QUERY_IDS)}

assert len(DEV_QUERY_IDS) == 6666
assert dev_query_embeddings.shape == (6666, DIM)
assert all(q in DEV_QUERY_INDEX for q in FIT_IDS)
assert all(q in DEV_QUERY_INDEX for q in VAL_IDS)

PQ_PATH = RUN_V018 / "fever-e5-small-v2-ivfpq-nlist4096-m32-nbits8.faiss"
SQ_PATH = RUN_V018 / "fever-e5-small-v2-ivfsq8-nlist4096.faiss"

print("Loading PQ32...")
pq32 = faiss.read_index(str(PQ_PATH))
print("Loading SQ8...")
sq8 = faiss.read_index(str(SQ_PATH))

EXPECTED_DOCS = 5_416_568
assert pq32.ntotal == EXPECTED_DOCS
assert sq8.ntotal == EXPECTED_DOCS

print("Query embeddings:", dev_query_embeddings.shape)
print("PQ32 ntotal:", f"{pq32.ntotal:,}")
print("SQ8 ntotal:", f"{sq8.ntotal:,}")
print("RESTORE — PASS")

In [ ]:
# Cell 4 — Restore exact v0.18 corpus row space + DEV qrels
CORPUS_IDS_PATH = RUN_V018 / "corpus_doc_ids.txt"
CORPUS_MEMMAP_PATH = RUN_V018 / "corpus_embeddings.float16.memmap"
DEV_QRELS_PATH = RAW_FEVER / "qrels" / "dev.tsv"

assert CORPUS_IDS_PATH.is_file()
assert CORPUS_MEMMAP_PATH.is_file()
assert DEV_QRELS_PATH.is_file()

qrels_df = pd.read_csv(DEV_QRELS_PATH, sep="\t")
qcol = next(c for c in ["query-id", "query_id", "qid"] if c in qrels_df.columns)
dcol = next(c for c in ["corpus-id", "corpus_id", "doc_id"] if c in qrels_df.columns)
scol = next((c for c in ["score", "relevance", "rel"] if c in qrels_df.columns), None)

qrels_df[qcol] = qrels_df[qcol].astype(str)
qrels_df[dcol] = qrels_df[dcol].astype(str)
qrels_df = qrels_df[qrels_df[qcol].isin(DEV_QUERY_IDS)].copy()
if scol is not None:
    qrels_df[scol] = pd.to_numeric(qrels_df[scol], errors="coerce")
    qrels_df = qrels_df[qrels_df[scol] > 0].copy()

needed_doc_ids = set(qrels_df[dcol])
doc_to_row = {}

with open(CORPUS_IDS_PATH, "r", encoding="utf-8") as f:
    for row, line in enumerate(tqdm(f, total=EXPECTED_DOCS, desc="Mapping qrel docs")):
        doc_id = line.rstrip("\n")
        if doc_id in needed_doc_ids:
            doc_to_row[doc_id] = row

missing_docs = needed_doc_ids - set(doc_to_row)
assert not missing_docs, list(missing_docs)[:20]

QRELS = defaultdict(set)
for _, r in qrels_df.iterrows():
    QRELS[str(r[qcol])].add(int(doc_to_row[str(r[dcol])]))

assert all(q in QRELS and len(QRELS[q]) > 0 for q in VAL_IDS)

corpus_embeddings = np.memmap(
    CORPUS_MEMMAP_PATH,
    dtype=np.float16,
    mode="r",
    shape=(EXPECTED_DOCS, DIM),
)

print("Mapped relevant docs:", len(doc_to_row))
print("Corpus memmap:", corpus_embeddings.shape, corpus_embeddings.dtype)
print("QRELS + CORPUS ROW SPACE — PASS")

In [ ]:
# Cell 5 — Frozen 44-policy grid and trajectory helpers
ALPHAS = [0.1, 0.3, 0.5, 0.7]
MEAN_K = [5, 20, 50]
SOFTMAX_K = [5, 20]
TEMPERATURES = [0.05, 0.1, 0.2, 0.5]

POLICIES = []
for alpha in ALPHAS:
    for k in MEAN_K:
        POLICIES.append({
            "method": "mean",
            "alpha": float(alpha),
            "k": int(k),
            "temperature": np.nan,
            "config_key": f"mean-k{k}-a{str(alpha).replace('.', 'p')}-tnone",
        })
    for k in SOFTMAX_K:
        for tau in TEMPERATURES:
            POLICIES.append({
                "method": "softmax",
                "alpha": float(alpha),
                "k": int(k),
                "temperature": float(tau),
                "config_key": (
                    f"softmax-k{k}-a{str(alpha).replace('.', 'p')}-"
                    f"t{str(tau).replace('.', 'p')}"
                ),
            })

assert len(POLICIES) == 44

def norm_vec(x, eps=1e-12):
    x = np.asarray(x, dtype=np.float32)
    return x / max(float(np.linalg.norm(x)), eps)

def slope(y):
    y = np.asarray(y, dtype=np.float64)
    x = np.arange(len(y), dtype=np.float64)
    return float(np.polyfit(x, y, 1)[0])

def jacdist(a, b):
    A = set(map(int, a))
    B = set(map(int, b))
    return 1.0 - len(A & B) / max(1, len(A | B))

def ndcg(ids, relevant, k=UTILITY_K):
    ids = np.asarray(ids, dtype=np.int64)[:k]
    gains = np.asarray([1.0 if int(i) in relevant else 0.0 for i in ids], dtype=np.float64)
    discounts = 1.0 / np.log2(np.arange(2, len(ids) + 2))
    dcg = float(np.sum(gains * discounts))
    m = min(k, len(relevant))
    if m == 0:
        return 0.0
    idcg = float(np.sum(1.0 / np.log2(np.arange(2, m + 2))))
    return dcg / idcg

def search_sq8(q, nprobe):
    sq8.nprobe = int(nprobe)
    q = norm_vec(q)[None, :]
    scores, ids = sq8.search(q, TOP_RETRIEVE)
    valid = ids[0] >= 0
    return scores[0][valid], ids[0][valid]

def fetch_docs(rows):
    rows = np.asarray(rows, dtype=np.int64)
    x = np.asarray(corpus_embeddings[rows], dtype=np.float32)
    x /= np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)
    return x

def feedback_vector(scores, ids, policy):
    k = int(policy["k"])
    ids = np.asarray(ids[:k], dtype=np.int64)
    scores = np.asarray(scores[:k], dtype=np.float64)
    docs = fetch_docs(ids)

    if policy["method"] == "mean":
        f = docs.mean(axis=0)
    else:
        tau = float(policy["temperature"])
        z = scores / tau
        z -= z.max()
        w = np.exp(z)
        w /= w.sum()
        f = (docs * w[:, None]).sum(axis=0)
    return norm_vec(f)

def update_state(q0, feedback, alpha):
    return norm_vec((1.0 - float(alpha)) * q0 + float(alpha) * feedback)

print("Policies:", len(POLICIES))
print("HELPERS — PASS")

In [ ]:
# Cell 6 — Nprobe-only coupled trajectory implementation
def run_nprobe_pair(qid, policy):
    q0 = norm_vec(dev_query_embeddings[DEV_QUERY_INDEX[qid]])
    qL = q0.copy()
    qH = q0.copy()
    rel = QRELS[qid]
    rows = []

    for t in range(MAX_ROUNDS + 1):
        sL, iL = search_sq8(qL, MATCHED_NPROBE)
        sH, iH = search_sq8(qH, HIGH_NPROBE)

        if min(len(iL), len(iH)) < max(int(policy["k"]), UTILITY_K):
            raise RuntimeError(f"Insufficient results qid={qid} policy={policy}")

        uL = ndcg(iL, rel)
        uH = ndcg(iH, rel)

        rows.append({
            "query_id": str(qid),
            "mechanism": "nprobe_severity_matched",
            "iteration": int(t),
            "method": policy["method"],
            "alpha": float(policy["alpha"]),
            "k": int(policy["k"]),
            "temperature": (
                float(policy["temperature"])
                if not pd.isna(policy["temperature"]) else np.nan
            ),
            "config_key": policy["config_key"],
            "query_state_distance": 1.0 - float(np.dot(norm_vec(qL), norm_vec(qH))),
            "candidate_jaccard_distance": jacdist(iL[:TOP_RETRIEVE], iH[:TOP_RETRIEVE]),
            "utility_low": uL,
            "utility_high": uH,
            "signed_utility_gap": uH - uL,
            "abs_utility_gap": abs(uH - uL),
        })

        if t == MAX_ROUNDS:
            break

        fL = feedback_vector(sL, iL, policy)
        fH = feedback_vector(sH, iH, policy)
        qL = update_state(q0, fL, policy["alpha"])
        qH = update_state(q0, fH, policy["alpha"])

    return rows

def endpoint_df(traj):
    out = []
    group_cols = [
        "query_id", "mechanism", "method", "alpha", "k", "temperature", "config_key"
    ]
    for keys, g in traj.groupby(group_cols, dropna=False, sort=False):
        g = g.sort_values("iteration")
        out.append({
            "query_id": keys[0],
            "mechanism": keys[1],
            "method": keys[2],
            "alpha": keys[3],
            "k": keys[4],
            "temperature": keys[5],
            "config_key": keys[6],
            "H1_slope": slope(g["query_state_distance"]),
            "H2_slope": slope(g["candidate_jaccard_distance"]),
            "H3_abs_slope": slope(g["abs_utility_gap"]),
            "H3_signed_slope": slope(g["signed_utility_gap"]),
            "final_signed_gap": float(g["signed_utility_gap"].iloc[-1]),
        })
    return pd.DataFrame(out)

smoke_qid = FIT_IDS[0]
for p in [POLICIES[0], POLICIES[-1]]:
    s = pd.DataFrame(run_nprobe_pair(smoke_qid, p))
    assert len(s) == MAX_ROUNDS + 1
    assert np.isfinite(
        s[["query_state_distance","candidate_jaccard_distance",
           "utility_low","utility_high","signed_utility_gap","abs_utility_gap"]].to_numpy()
    ).all()

print("FIT-ONLY IMPLEMENTATION SMOKE — PASS")
print("Validation trajectories untouched.")

## Before Cell 7

At this point the implementation smoke test used **FIT only**. The selected `nprobe=8` is already frozen by the earlier v0.25 protocol.

Cell 7 is the confirmatory validation sweep. It writes one checkpoint every 10 validation queries to Drive and resumes automatically from existing checkpoints.

In [ ]:
# Cell 7 — Resumable validation sweep: ONLY severity-matched nprobe branch
RUN_DIR = OUT / "validation-nprobe-matched"
RUN_DIR.mkdir(parents=True, exist_ok=True)

expected_rows = len(VAL_IDS) * len(POLICIES) * (MAX_ROUNDS + 1)

print("VALIDATION QUERIES:", len(VAL_IDS))
print("POLICIES:", len(POLICIES))
print("EXPECTED TRAJECTORY ROWS:", f"{expected_rows:,}")
print("CHECKPOINT DIR:", RUN_DIR)

for start in range(0, len(VAL_IDS), CHECKPOINT_EVERY_QUERIES):
    stop = min(start + CHECKPOINT_EVERY_QUERIES, len(VAL_IDS))
    cp = RUN_DIR / f"traj_{start:05d}_{stop:05d}.parquet"

    if cp.exists():
        print("skip", cp.name)
        continue

    rows = []
    t0 = time.perf_counter()

    for qi in range(start, stop):
        qid = VAL_IDS[qi]
        for policy in POLICIES:
            rows.extend(run_nprobe_pair(qid, policy))

    df_cp = pd.DataFrame(rows)
    tmp = cp.with_suffix(".tmp.parquet")
    df_cp.to_parquet(tmp, index=False)
    os.replace(tmp, cp)

    print(
        f"wrote {cp.name}: {len(df_cp):,} rows | "
        f"{time.perf_counter() - t0:.1f}s"
    )

parts = sorted(RUN_DIR.glob("traj_*.parquet"))
assert parts

traj_np = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)

assert len(traj_np) == expected_rows, (len(traj_np), expected_rows)
assert traj_np["query_id"].nunique() == len(VAL_IDS)

NP_TRAJ_PATH = OUT / "v025_validation_nprobe_matched_trajectories.parquet"
traj_np.to_parquet(NP_TRAJ_PATH, index=False)

np_endpoints = endpoint_df(traj_np)
NP_ENDPOINT_PATH = OUT / "v025_validation_nprobe_matched_endpoints.parquet"
np_endpoints.to_parquet(NP_ENDPOINT_PATH, index=False)

print("Trajectory rows:", f"{len(traj_np):,}")
print("Endpoint rows:", f"{len(np_endpoints):,}")
print("VALIDATION NPROBE SWEEP — COMPLETE")

In [ ]:
# Cell 8 — Load existing v0.18 representation endpoints and normalize schema
REP_ENDPOINT_PATH = RUN_V018 / "v018_validation_endpoints.parquet"
assert REP_ENDPOINT_PATH.is_file(), REP_ENDPOINT_PATH

rep_endpoints = pd.read_parquet(REP_ENDPOINT_PATH)

print("v0.18 representation endpoint columns:")
print(rep_endpoints.columns.tolist())
print("rows:", len(rep_endpoints))
print("queries:", rep_endpoints["query_id"].astype(str).nunique())

rename_map = {}
candidates = {
    "H1_slope": ["H1_slope", "H1", "query_state_slope"],
    "H2_slope": ["H2_slope", "H2", "candidate_jaccard_slope"],
    "H3_abs_slope": ["H3_abs_slope", "H3_abs", "H3_slope", "abs_utility_gap_slope"],
    "H3_signed_slope": ["H3_signed_slope", "H3_signed", "signed_utility_gap_slope"],
}
for target, opts in candidates.items():
    if target not in rep_endpoints.columns:
        src = next((x for x in opts if x in rep_endpoints.columns), None)
        if src is not None:
            rename_map[src] = target

rep_endpoints = rep_endpoints.rename(columns=rename_map)

required = ["query_id", "method", "alpha", "k", "H1_slope", "H2_slope", "H3_abs_slope"]
missing = [c for c in required if c not in rep_endpoints.columns]
assert not missing, f"Missing representation endpoint columns: {missing}"

if "temperature" not in rep_endpoints.columns:
    rep_endpoints["temperature"] = np.nan
if "config_key" not in rep_endpoints.columns:
    rep_endpoints["config_key"] = (
        rep_endpoints["method"].astype(str) + "|a=" + rep_endpoints["alpha"].astype(str)
        + "|k=" + rep_endpoints["k"].astype(str)
        + "|t=" + rep_endpoints["temperature"].astype(str)
    )
if "H3_signed_slope" not in rep_endpoints.columns:
    rep_endpoints["H3_signed_slope"] = np.nan

rep_endpoints["query_id"] = rep_endpoints["query_id"].astype(str)
rep_endpoints["mechanism"] = "representation"

assert rep_endpoints["query_id"].nunique() == len(VAL_IDS)
assert set(rep_endpoints["query_id"].unique()) == set(VAL_IDS)

print("REPRESENTATION ENDPOINT RESTORE — PASS")

In [ ]:
# Cell 9 — Query-level mechanism comparison + 10k paired bootstrap
MEASURES = ["H1_slope", "H2_slope", "H3_abs_slope", "H3_signed_slope"]

def query_level_average(df, mechanism):
    use = [c for c in MEASURES if c in df.columns and not df[c].isna().all()]
    q = df.groupby("query_id", as_index=False)[use].mean()
    q["mechanism"] = mechanism
    return q

rep_q = query_level_average(rep_endpoints, "representation")
np_q = query_level_average(np_endpoints, "nprobe_severity_matched")

assert set(rep_q["query_id"]) == set(np_q["query_id"]) == set(VAL_IDS)

display(rep_q[["H1_slope","H2_slope","H3_abs_slope"]].mean().to_frame("representation").T)
display(np_q[["H1_slope","H2_slope","H3_abs_slope"]].mean().to_frame("nprobe_matched").T)

rng = np.random.default_rng(SEED + 2501)

def bootstrap_mean(x, reps=BOOTSTRAP_REPS):
    x = np.asarray(x, dtype=np.float64)
    n = len(x)
    point = float(x.mean())
    boots = np.empty(reps, dtype=np.float64)
    for b in range(reps):
        idx = rng.integers(0, n, size=n)
        boots[b] = float(x[idx].mean())
    lo, hi = np.quantile(boots, [0.025, 0.975])
    return point, float(lo), float(hi), n

rep_h3 = rep_q.set_index("query_id").loc[VAL_IDS, "H3_abs_slope"].to_numpy()
np_h3 = np_q.set_index("query_id").loc[VAL_IDS, "H3_abs_slope"].to_numpy()
paired_diff = rep_h3 - np_h3

rep_stat = bootstrap_mean(rep_h3)
np_stat = bootstrap_mean(np_h3)
diff_stat = bootstrap_mean(paired_diff)

primary_summary = pd.DataFrame([
    {"estimand":"representation_H3abs", "mean":rep_stat[0],
     "ci95_low":rep_stat[1], "ci95_high":rep_stat[2], "n_queries":rep_stat[3]},
    {"estimand":"matched_nprobe_H3abs", "mean":np_stat[0],
     "ci95_low":np_stat[1], "ci95_high":np_stat[2], "n_queries":np_stat[3]},
    {"estimand":"representation_minus_matched_nprobe_H3abs", "mean":diff_stat[0],
     "ci95_low":diff_stat[1], "ci95_high":diff_stat[2], "n_queries":diff_stat[3]},
])

PRIMARY_SUMMARY_PATH = OUT / "v025_severity_matched_primary_query_bootstrap.csv"
primary_summary.to_csv(PRIMARY_SUMMARY_PATH, index=False)

display(primary_summary.round(6))

In [ ]:
# Cell 10 — Family-balanced robustness audit (mean=50%, softmax=50%)
def family_query_values(df):
    x = df.copy()
    x["query_id"] = x["query_id"].astype(str)

    fam = (
        x.groupby(["query_id", "method"], as_index=False)["H3_abs_slope"]
        .mean()
    )

    piv = fam.pivot(index="query_id", columns="method", values="H3_abs_slope")
    assert {"mean", "softmax"}.issubset(piv.columns)

    piv["family_balanced_H3abs"] = 0.5 * piv["mean"] + 0.5 * piv["softmax"]
    return piv

rep_f = family_query_values(rep_endpoints)
np_f = family_query_values(np_endpoints)

common = sorted(set(rep_f.index) & set(np_f.index))
assert set(common) == set(VAL_IDS)

rep_bal = rep_f.loc[common, "family_balanced_H3abs"].to_numpy()
np_bal = np_f.loc[common, "family_balanced_H3abs"].to_numpy()
diff_bal = rep_bal - np_bal

rep_bal_stat = bootstrap_mean(rep_bal)
np_bal_stat = bootstrap_mean(np_bal)
diff_bal_stat = bootstrap_mean(diff_bal)

family_summary = pd.DataFrame([
    {"estimand":"representation_family_balanced_H3abs",
     "mean":rep_bal_stat[0], "ci95_low":rep_bal_stat[1],
     "ci95_high":rep_bal_stat[2], "n_queries":rep_bal_stat[3]},
    {"estimand":"matched_nprobe_family_balanced_H3abs",
     "mean":np_bal_stat[0], "ci95_low":np_bal_stat[1],
     "ci95_high":np_bal_stat[2], "n_queries":np_bal_stat[3]},
    {"estimand":"representation_minus_matched_nprobe_family_balanced_H3abs",
     "mean":diff_bal_stat[0], "ci95_low":diff_bal_stat[1],
     "ci95_high":diff_bal_stat[2], "n_queries":diff_bal_stat[3]},
])

FAMILY_SUMMARY_PATH = OUT / "v025_severity_matched_family_balanced_bootstrap.csv"
family_summary.to_csv(FAMILY_SUMMARY_PATH, index=False)

display(family_summary.round(6))

In [ ]:
# Cell 11 — Frozen validation gate and final report
gate = {
    "status": "ARC_V025_SEVERITY_MATCHED_VALIDATION_ANALYZED",
    "protocol_sha256": PROTOCOL_SHA,
    "dataset": "FEVER",
    "encoder": "intfloat/e5-small-v2",
    "n_validation_queries": len(VAL_IDS),
    "fit_representation_gap": float(protocol["fit_representation_gap"]),
    "fit_matched_search_gap": float(protocol["fit_matched_search_gap"]),
    "fit_relative_gap_mismatch": float(protocol["fit_relative_gap_mismatch"]),
    "matched_nprobe_low": MATCHED_NPROBE,
    "representation_H3abs": {
        "mean": rep_stat[0],
        "ci95": [rep_stat[1], rep_stat[2]],
        "direction_positive": bool(rep_stat[0] > 0),
    },
    "matched_nprobe_H3abs": {
        "mean": np_stat[0],
        "ci95": [np_stat[1], np_stat[2]],
        "direction_negative": bool(np_stat[0] < 0),
    },
    "paired_rep_minus_nprobe_H3abs": {
        "mean": diff_stat[0],
        "ci95": [diff_stat[1], diff_stat[2]],
        "positive": bool(diff_stat[0] > 0),
        "ci_excludes_zero_positive": bool(diff_stat[1] > 0),
    },
    "family_balanced_paired_rep_minus_nprobe_H3abs": {
        "mean": diff_bal_stat[0],
        "ci95": [diff_bal_stat[1], diff_bal_stat[2]],
        "positive": bool(diff_bal_stat[0] > 0),
        "ci_excludes_zero_positive": bool(diff_bal_stat[1] > 0),
    },
    "frozen_sign_map_matches": bool(rep_stat[0] > 0 and np_stat[0] < 0),
    "severity_matched_mechanism_contrast_supported": bool(
        diff_stat[1] > 0 and diff_bal_stat[1] > 0
    ),
    "negative_null_or_reversal_retained": True,
    "validation_retuning_performed": False,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
}

REPORT_PATH = OUT / "v025_severity_matched_validation_final_report.json"
REPORT_PATH.write_text(json.dumps(gate, indent=2, sort_keys=True))

print(json.dumps(gate, indent=2))
print("\nFINAL REPORT:", REPORT_PATH)

In [ ]:
# Cell 12 — Artifact hashes
artifact_paths = [
    PROTOCOL_PATH,
    NP_TRAJ_PATH,
    NP_ENDPOINT_PATH,
    PRIMARY_SUMMARY_PATH,
    FAMILY_SUMMARY_PATH,
    REPORT_PATH,
]

hash_rows = []
for p in artifact_paths:
    if p.exists():
        hash_rows.append({
            "file": p.name,
            "bytes": p.stat().st_size,
            "sha256": sha256_file(p),
        })

hash_df = pd.DataFrame(hash_rows)
HASH_PATH = OUT / "V025_SEVERITY_MATCHED_VALIDATION_ARTIFACT_SHA256.csv"
hash_df.to_csv(HASH_PATH, index=False)

display(hash_df)

print("=" * 88)
print("ARC-v0.25 SEVERITY-MATCHED VALIDATION — COMPLETE")
print("OUT:", OUT)
print("sign map matches:", gate["frozen_sign_map_matches"])
print("paired contrast supported:", gate["severity_matched_mechanism_contrast_supported"])
print("=" * 88)

## Send back

After Cell 12 completes, send either the executed notebook or paste the outputs from:

- Cell 9: `v025_severity_matched_primary_query_bootstrap.csv`
- Cell 10: `v025_severity_matched_family_balanced_bootstrap.csv`
- Cell 11: final JSON gate

The most important confirmatory result is the paired query-level `representation_minus_matched_nprobe_H3abs` and its family-balanced counterpart.

A negative or null outcome must be retained unchanged.